https://judge.nitro-ai.org/competitions/roai-2025/simulare-ojia/1/view?participant_name=ecaterina

#setup + imports


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

In [ ]:
train_data = pd.read_csv('/content/train_data.csv')
test_data = pd.read_csv('/content/test_data.csv')

In [ ]:
y = train_data.on_time
X = train_data.drop(['on_time'], axis=1)

#subtask 1 + 2

In [ ]:
answers1 = round(test_data['traffic_level'].mean(), 2)

subtask1 = pd.DataFrame({
    "subtaskID": 1,
    "datapointID": 1,
    "answer": answers1
}, index=[0])

In [ ]:
answers2 = round(test_data['traffic_level'].std(), 2)

subtask2 = pd.DataFrame({
    "subtaskID": 2,
    "datapointID": 1,
    "answer": answers2
}, index=[0])

# data exploration

In [ ]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 100 non-null    int64  
 1   distance_km        100 non-null    float64
 2   package_weight_kg  100 non-null    float64
 3   traffic_level      100 non-null    int64  
 4   on_time            100 non-null    int64  
dtypes: float64(2), int64(3)
memory usage: 4.0 KB


In [ ]:
print(train_data.isna().sum().sum(), test_data.isna().sum().sum())
print("duplicate rows:", train_data.duplicated().sum())
train_data.nunique()

0 0
duplicate rows: 0


,0
id,100
distance_km,100
package_weight_kg,100
traffic_level,14
on_time,2


# data cleaning


In [ ]:
def clean_input(data):
  data = data.copy()
  data = data.drop(columns='id')
  return data

In [ ]:
X_clean = clean_input(X)
test_clean = clean_input(test_data)

print(X_clean.dtypes)
print(list(X_clean.columns) == list(test_clean.columns))

distance_km          float64
package_weight_kg    float64
traffic_level          int64
dtype: object
True


# model + validare

In [ ]:
def eval(model, X_clean, y):
  kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
  scores = []
  for fitrows, valrows in kf.split(X_clean, y):
    X_fit = X_clean.iloc[fitrows]
    y_fit = y.iloc[fitrows]
    X_val = X_clean.iloc[valrows]
    y_val = y.iloc[valrows]

    model.fit(X_fit, y_fit)
    preds = model.predict(X_val)
    score = accuracy_score(y_val, preds)
    scores.append(score)

  print('fold scores:', [round(s, 2) for s in scores])
  print('mean:', round(np.mean(scores), 3), ' spread:', round(np.std(scores), 3))

In [ ]:
eval(DummyClassifier(strategy='most_frequent'), X_clean, y)

fold scores: [0.45, 0.5, 0.5, 0.5, 0.5]
mean: 0.49  spread: 0.02


In [ ]:
eval(HistGradientBoostingClassifier(loss='log_loss', random_state=0), X_clean, y)

fold scores: [1.0, 1.0, 1.0, 1.0, 1.0]
mean: 1.0  spread: 0.0


#prediction

In [ ]:
model = HistGradientBoostingClassifier(loss='log_loss', random_state=0)
model.fit(X_clean, y)
preds = model.predict(test_clean)
print('mean:', round(np.mean(preds), 3))

mean: 0.48


In [ ]:
subtask3 = pd.DataFrame({
    "subtaskID": 3,
    "datapointID": test_data["id"],
    "answer": [str(p) for p in preds]
})

In [ ]:
submission = pd.concat([subtask1, subtask2, subtask3], ignore_index=True)
submission.to_csv('submission.csv', index=False)